KEYWORDS


In [ ]:
import os
import yake

In [ ]:
# Instantier l'extracteur de mots clés
kw_extractor = yake.KeywordExtractor(lan="fr", top=50)
kw_extractor

In [ ]:
# Lister les Fichiers
data_path = "../../data/txt/"
files = [f for f in os.listdir(data_path) if f.endswith('.txt')]

In [ ]:
# Imprimer le nombre de fichiers identifiés
len(files)

In [ ]:
#choisir année à analyser 
year = 1965
#séléctionner fichiers de cette année 
files_year = [f for f in files if f"_{year}-" in f]

print(f"Nombre de fichiers pour {year} : {len(files_year)}")
print(files_year[:10])

In [ ]:
# Choisir un fichier de 1965
this_file = files_year[0]
this_file

In [ ]:
# Récupérer le texte du fichier
text = open(os.path.join(data_path, this_file), 'r', encoding='utf-8').read()
text[:500]

In [ ]:
# Extraire les mots clés de ce texte
keywords = kw_extractor.extract_keywords(text)
keywords

In [ ]:
# Ne garder que les bigrammes
kept = []
for kw, score in keywords:
    words = kw.split()
    if len(words) == 2:
        kept.append(kw)
kept

In [ ]:
for f in sorted(files_year):
    text = open(os.path.join(data_path, f), 'r', encoding="utf-8").read()
    keywords = kw_extractor.extract_keywords(text)
    kept = []
    for kw, score in keywords:
        words = kw.split()
        if len(words) == 2:
            kept.append(kw)
    print(f"{f} mentions these keywords: {', '.join(kept)}...")

NUAGE DE MOTS


In [ ]:
from collections import Counter
from wordcloud import WordCloud
import os
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from IPython.display import Image

In [ ]:
# Stopwords (Idem que dans s1)
sw = stopwords.words("french")
sw += ["les", "plus", "cette", "fait", "faire", "être", "deux", "comme", "dont", "tout",
       "ils", "bien", "sans", "peut", "tous", "après", "ainsi", "donc", "cet", "sous",
       "celle", "entre", "encore", "toutes", "pendant", "moins", "dire", "cela", "non",
       "faut", "trois", "aussi", "dit", "avoir", "doit", "contre", "depuis", "autres",
       "van", "het", "autre", "jusqu", "ville", "rossel", "dem"]
# première itération : enlever mots vides identifiés dans les fréquents
sw += ["rue","tél","ans","très","brux","heures"]
#deuxième itération 
sw += ["ecrire", "soir", "app", "bon", "gar", "leurs", "ecr"]
#dernier ajustement
sw += ["téléph", "prés", "cuis", "etc", "terr", "conf", "mod", "jard","sem","ares","vitae", "aff","AFF","tel","TEL", "trav", "TRAV", "bur","BUR", "cependant", "alors", "près","celui","ceux", "liv"]
# Jours de la semaine
sw += ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
# Mois de l'année
sw += ["janvier", "février", "mars", "avril", "mai", "juin",
"juillet", "août", "septembre", "octobre", "novembre", "décembre"]
sw = set(sw)

In [ ]:
# Choisir une année
year = 1965

In [ ]:
# Lister les fichiers de cette année
data_path = '../../data'
txt_path = '../../data/txt'
txts = [f for f in os.listdir(txt_path) if os.path.isfile(os.path.join(txt_path, f)) and str(year) in f]
len(txts)

In [ ]:
# Stocker le contenu de ces fichiers dans une liste
content_list = []
for txt in txts:
    with open(os.path.join(txt_path, txt), 'r', encoding='utf-8') as f:
        content_list.append(f.read())

In [ ]:
# Ecrire tout le contenu dans un fichier temporaire
temp_path = '../../data/tmp'
if not os.path.exists(temp_path):
    os.mkdir(temp_path)
with open(os.path.join(temp_path, f'{year}.txt'), 'w', encoding='utf-8') as f:
    f.write(' '.join(content_list))

In [ ]:
# Imprimer le contenu du fichier et constater les "déchets"
with open(os.path.join(temp_path, f'{year}.txt'), 'r', encoding='utf-8') as f:
    before = f.read()

before[:500]

In [ ]:
def clean_text(year, folder=None):
    if folder is None:
        input_path = f"{year}.txt"
        output_path = f"{year}_clean.txt"
    else:
        input_path = f"{folder}/{year}.txt"
        output_path = f"{folder}/{year}_clean.txt"
    output = open(output_path, "w", encoding='utf-8')
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()
        words = nltk.wordpunct_tokenize(text)
        kept = [w.upper() for w in words if len(w) > 2 and w.isalpha() and w.lower() not in sw]
        kept_string = " ".join(kept)
        output.write(kept_string)
    return f'Output has been written in {output_path}!'

In [ ]:
clean_text(year, folder=temp_path)

In [ ]:
# Vérifier le résultat
with open(os.path.join(temp_path, f'{year}_clean.txt'), 'r', encoding='utf-8') as f:
    after = f.read()

after[:500]

In [ ]:
frequencies = Counter(after.split())
print(frequencies.most_common(100))

In [ ]:
cloud = WordCloud(width=2000, height=1000, background_color='white').generate_from_frequencies(frequencies)
cloud.to_file(os.path.join(temp_path, f"{year}.png"))
Image(filename=os.path.join(temp_path, f"{year}.png"))

RECONNAISSANCE ENTITES NOMMEES


In [ ]:
from collections import defaultdict
import os
import spacy
from spacy.lang.fr.examples import sentences

In [ ]:
nlp = spacy.load('fr_core_news_md')

In [ ]:
# Charger le corpus 1965
year = 1965
data_path = "../../data/txt/"

files = [f for f in os.listdir(data_path) if f"_{year}-" in f and f.endswith(".txt")]

print(len(files))

In [ ]:
# créer les dictionnaires pour compter les entités
people = defaultdict(int)
locations = defaultdict(int)
organizations = defaultdict(int)

# parcourir les articles de 1965
for f in files:
    text = open(os.path.join(data_path, f), encoding="utf-8").read()
    doc = nlp(text)

    # Compter les entités identifiées
    for ent in doc.ents:
        if len(ent.text) > 3:
            if ent.label_ == "PER":
                people[ent.text] += 1
            elif ent.label_ == "LOC":
                locations[ent.text] += 1
            elif ent.label_ == "ORG":
                organizations[ent.text] += 1

In [ ]:
# Trier et imprimer les personnes

sorted_people = sorted(people.items(), key=lambda kv: kv[1], reverse=True)

for person, freq in sorted_people[:50]:
    print(f"{person} apparait {freq} fois dans le corpus")

In [ ]:
# Trier et imprimer les lieux
sorted_locations = sorted(locations.items(), key=lambda kv: kv[1], reverse=True)

for location, freq in sorted_locations[:20]:
    print(f"{location} apparait {freq} fois dans le corpus")

In [ ]:
# Trier et imprimer les organisations
sorted_organizations = sorted(organizations.items(), key=lambda kv: kv[1], reverse=True)

for organization, freq in sorted_organizations[:20]:
    print(f"{organization} apparait {freq} fois dans le corpus")

ANALYSE DE SENTIMENT


In [ ]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

In [ ]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")

In [ ]:
#Phrase 1
get_sentiment("Le marché est calme et sans relief.")

In [ ]:
#Phrase 2
get_sentiment("Grâce à une excellente défense, l’équipe visiteuse résista fort joliment, surtout par le fait de De Greef")

In [ ]:
#Phrase 3
get_sentiment("Le problème n'était pas facile à résoudre.")

In [ ]:
#Phrase 4
get_sentiment("Cette nouvelle réglementation a été approuvée par le président Nasser.")

In [ ]:
#Phrase 5
get_sentiment("La première période fut caractérisée par une imprécision lamentable.")

In [ ]:
#Phrase 6
get_sentiment("Victoire méritée de Linthout.")

In [ ]:
#Phrase 7
get_sentiment("Les dernières minutes étaient crispantes.")

In [ ]:
#Phrase 8
get_sentiment("On constate une baisse de 15 à 20 centimes par oeuf.")

In [ ]:
#Phrase 9
get_sentiment("En 1964, la fièvre aphteuse a été enregistrée dans 18 départements et 55 exploitations, soit un total de 55 foyers.")

In [ ]:
#Phrase 10
get_sentiment("Il y a encore une légère avance en Anilin, mais Bayer vacille, de même que Erdoel.")